# Aula 11 - Notebook: Modelagem Topológica de Tubulações, Calhas e Instrumentos como Dígrafos Ponderados

**Projeto:** SCADA-Core Automática — GRUPO 4  
**Unidade:** Planta Automatizada de Seleção e Classificação Óptica de Grãos com Separação Pneumática Tripla  
**Engenharia de Automação e Controle** — ECAA08  

---

### 1. Objetivos do Notebook
Neste notebook implementamos a classe base `GrafoTubulacao` para formalizar a malha de transporte mecânico, dutos gravitacionais, calhas e ejetores pneumáticos da planta como um **Grafo Dirigido e Ponderado (Dígrafo)**:

$$G = (V, E, W)$$

Onde:
* $V$: Conjunto de 14 nós físicos e lógicos (funil, calha vibratória, esteira, pesagem, túnel de visão, separadores e silos);
* $E$: Conjunto de 14 arestas orientadas (dutos, correia tracionada, sopro de ar comprimido e reciclo);
* $W$: Matriz de ponderação que quantifica os comprimentos físicos reais em metros ($L\text{ [m]}$) e tempos de trânsito.

### 2. Funções Auxiliares de Formatação Tabular e Matricial em ASCII

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII pura com suporte a infinitos."""
    larguras = [max(len(str(r)), 7) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "inf" if v == float('inf') else f"{v:.1f}" if isinstance(v, float) else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)


### 3. Implementação Orientada a Objetos: Classe `GrafoTubulacao`

A classe `GrafoTubulacao` encapsula:
1. Dicionários bidirecionais de mapeamento Vértice $\leftrightarrow$ Índice;
2. Matriz de adjacência binária ($A \in \{0, 1\}^{n \times n}$);
3. Matriz de adjacência ponderada ($W \in (\mathbb{R}^+ \cup \{\infty\})^{n \times n}$);
4. Metadados de instrumentação ISA-5.1, bitolas e atuadores;
5. Validação automática do **Lema do Aperto de Mãos Dirigido**.

In [2]:
from typing import List, Dict, Any

class GrafoTubulacao:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0
        
        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_tubulacao(self, origem: str, destino: str, comprimento_m: float,
                           tag_valvula: str, meio_transporte: str = "Duto", bitola: str = "DN200"):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        
        self.arestas_detalhes.append({
            "Aresta": f"e{len(self.arestas_detalhes)+1}",
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Atuador / ISA": tag_valvula,
            "Meio Físico": meio_transporte,
            "Bitola / Dim": bitola
        })

    def obter_graus(self) -> List[Dict[str, Any]]:
        graus = []
        for i, v in enumerate(self.vertices):
            deg_out = sum(self.adj_binaria[i])
            deg_in = sum(self.adj_binaria[r][i] for r in range(self.n))
            balanco = "Balanceado" if deg_in == deg_out else ("Bifurcação" if deg_out > deg_in else "Sumidouro")
            graus.append({
                "Equipamento / Nó": v,
                "Grau Entrada (deg-)": deg_in,
                "Grau Saída (deg+)": deg_out,
                "Comportamento": balanco
            })
        return graus

    def validar_lema_aperto_maos(self) -> bool:
        soma_out = sum(sum(self.adj_binaria[i]) for i in range(self.n))
        soma_in = sum(sum(self.adj_binaria[r][i] for r in range(self.n)) for i in range(self.n))
        total_arestas = len(self.arestas_detalhes)
        print("Validação do Lema do Aperto de Mãos Dirigido:")
        print(f"-> Somatório deg+ : {soma_out}")
        print(f"-> Somatório deg- : {soma_in}")
        print(f"-> Total de Arestas |E|: {total_arestas}")
        assert soma_out == soma_in == total_arestas, "Erro: Inconsistência nos graus da rede!"
        print("-> Conclusão: Lema Verificado com Sucesso! Identidade estrita confirmada.")
        return True


### 4. Configuração e Inicialização da Rede da Planta de Grãos (Grupo 4)

Cadastramos os 14 equipamentos da planta e todos os 14 segmentos de condução, calhas, ejetores ultrarrápidos e linha de reciclo.

In [3]:
nos_processo = [
    "FN-101_Recepcao", "ALIM-101_Calha", "EST-201_Entrada", "BAL-301_Pesagem",
    "VIS-401_Inspecao", "DIV-501_Decisao", "EJE-602_EstacaoB", "EJE-603_EstacaoC",
    "CALHA-602_Secundario", "CALHA-603_Rejeito", "CALHA-701_Terminal",
    "SILO-701_Aprovados", "SILO-702_Secundario", "SILO-703_Rejeito"
]

rede = GrafoTubulacao(nos_processo)

# e1 a e5: Recepcao, alimentacao dosada, esteira de pesagem e inspecao optica
rede.adicionar_tubulacao("FN-101_Recepcao", "ALIM-101_Calha", 1.2, "XV-101", "Duto Vertical de Carga", "DN200")
rede.adicionar_tubulacao("ALIM-101_Calha", "EST-201_Entrada", 1.8, "c_ALIM", "Calha Vibratória Dosadora", "L=300mm")
rede.adicionar_tubulacao("EST-201_Entrada", "BAL-301_Pesagem", 2.5, "c_EST/ST-201", "Correia Transportadora", "L=400mm")
rede.adicionar_tubulacao("BAL-301_Pesagem", "VIS-401_Inspecao", 3.0, "WT-301/FT-301", "Correia Transportadora", "L=400mm")
rede.adicionar_tubulacao("VIS-401_Inspecao", "DIV-501_Decisao", 1.0, "XS-401/KSA-401", "Zona de Trigger Óptico", "L=400mm")

# e6 a e10: Estacoes de ejeção e continuidade da esteira
rede.adicionar_tubulacao("DIV-501_Decisao", "EJE-602_EstacaoB", 2.0, "ZC-602/p_POS602", "Correia Sincronizada", "L=400mm")
rede.adicionar_tubulacao("EJE-602_EstacaoB", "CALHA-602_Secundario", 0.8, "FY-602/ZSH-602", "Sopro Pneumático B", "DN15")
rede.adicionar_tubulacao("EJE-602_EstacaoB", "EJE-603_EstacaoC", 1.5, "ST-201", "Correia Contínua", "L=400mm")
rede.adicionar_tubulacao("EJE-603_EstacaoC", "CALHA-603_Rejeito", 0.8, "FY-603/ZSH-601", "Sopro Pneumático C", "DN15")
rede.adicionar_tubulacao("EJE-603_EstacaoC", "CALHA-701_Terminal", 2.2, "Descarga Inércia", "Correia Final Livre", "L=400mm")

# e11 a e14: Calhas gravitacionais para silos e linha de reciclo de graos comerciais
rede.adicionar_tubulacao("CALHA-701_Terminal", "SILO-701_Aprovados", 3.5, "XV-701", "Duto Gravitacional A", "DN250")
rede.adicionar_tubulacao("CALHA-602_Secundario", "SILO-702_Secundario", 4.0, "XV-702", "Duto Inclinado B", "DN200")
rede.adicionar_tubulacao("CALHA-603_Rejeito", "SILO-703_Rejeito", 3.2, "XV-703", "Duto Inclinado C", "DN200")
rede.adicionar_tubulacao("SILO-702_Secundario", "FN-101_Recepcao", 12.0, "XV-104", "Elevador de Reciclo/Reteste", "Canecas 6pol")


### 5. Execução, Relatórios Topológicos e Matrizes de Adjacência

In [4]:
print("=== TABELA DE DUTOS E CONEXÕES FÍSICAS DA PLANTA ===")
print(formatar_tabela(rede.arestas_detalhes))

print("\n=== GRAUS TOPOLÓGICOS DOS EQUIPAMENTOS ===")
print(formatar_tabela(rede.obter_graus()))

print("\n")
rede.validar_lema_aperto_maos()

siglas = ["FN101", "ALIM", "EST201", "BAL301", "VIS401", "DIV501", "EJE602", "EJE603", "CAL602", "CAL603", "CAL701", "SIL701", "SIL702", "SIL703"]
print("\n=== MATRIZ DE ADJACÊNCIA BINÁRIA A (Conexão Direta: 0 ou 1) ===")
print(formatar_matriz(rede.adj_binaria, rede.vertices, siglas))

print("\n=== MATRIZ DE ADJACÊNCIA PONDERADA W (Comprimento em Metros) ===")
print(formatar_matriz(rede.adj_pesos, rede.vertices, siglas))


=== TABELA DE DUTOS E CONEXÕES FÍSICAS DA PLANTA ===
Aresta | Origem               | Destino              | Comprimento (m) | Atuador / ISA    | Meio Físico                 | Bitola / Dim
-------+----------------------+----------------------+-----------------+------------------+-----------------------------+-------------
e1     | FN-101_Recepcao      | ALIM-101_Calha       | 1.2             | XV-101           | Duto Vertical de Carga      | DN200       
e2     | ALIM-101_Calha       | EST-201_Entrada      | 1.8             | c_ALIM           | Calha Vibratória Dosadora   | L=300mm     
e3     | EST-201_Entrada      | BAL-301_Pesagem      | 2.5             | c_EST/ST-201     | Correia Transportadora      | L=400mm     
e4     | BAL-301_Pesagem      | VIS-401_Inspecao     | 3.0             | WT-301/FT-301    | Correia Transportadora      | L=400mm     
e5     | VIS-401_Inspecao     | DIV-501_Decisao      | 1.0             | XS-401/KSA-401   | Zona de Trigger Óptico      | L=400mm     
e6

### 6. Análise de Rastreamento de Grãos e Tempo de Trânsito Dinâmico

Abaixo calculamos a distância e o tempo de propagação sincronizados com o encoder incremental `ST-201` ($v_{\text{esteira}} = 1{,}5\text{ m/s}$) para cada uma das três categorias de produto.

In [5]:
rotas = {
    "Categoria A (Nobre -> Silo A)": [
        "FN-101_Recepcao", "ALIM-101_Calha", "EST-201_Entrada", "BAL-301_Pesagem",
        "VIS-401_Inspecao", "DIV-501_Decisao", "EJE-602_EstacaoB", "EJE-603_EstacaoC",
        "CALHA-701_Terminal", "SILO-701_Aprovados"
    ],
    "Categoria B (Secundário -> Silo B)": [
        "FN-101_Recepcao", "ALIM-101_Calha", "EST-201_Entrada", "BAL-301_Pesagem",
        "VIS-401_Inspecao", "DIV-501_Decisao", "EJE-602_EstacaoB",
        "CALHA-602_Secundario", "SILO-702_Secundario"
    ],
    "Categoria C (Rejeito -> Silo C)": [
        "FN-101_Recepcao", "ALIM-101_Calha", "EST-201_Entrada", "BAL-301_Pesagem",
        "VIS-401_Inspecao", "DIV-501_Decisao", "EJE-602_EstacaoB", "EJE-603_EstacaoC",
        "CALHA-603_Rejeito", "SILO-703_Rejeito"
    ]
}

v_esteira = 1.5  # m/s (sinal ST-201)
relatorio_rotas = []

for nome_rota, seq in rotas.items():
    dist_total = 0.0
    for k in range(len(seq) - 1):
        u_idx = rede.v_to_idx[seq[k]]
        v_idx = rede.v_to_idx[seq[k+1]]
        dist_total += rede.adj_pesos[u_idx][v_idx]
    tempo_estimado = dist_total / v_esteira
    relatorio_rotas.append({
        "Rota Operacional": nome_rota,
        "Número de Nós": len(seq),
        "Distância Total (m)": f"{dist_total:.2f}",
        "Tempo Trânsito Estimado (s)": f"{tempo_estimado:.2f}"
    })

print("=== RELATÓRIO DINÂMICO DE ROTEAMENTO E TEMPOS DE PROPAGAÇÃO ===")
print(formatar_tabela(relatorio_rotas))


=== RELATÓRIO DINÂMICO DE ROTEAMENTO E TEMPOS DE PROPAGAÇÃO ===
Rota Operacional                   | Número de Nós | Distância Total (m) | Tempo Trânsito Estimado (s)
-----------------------------------+---------------+---------------------+----------------------------
Categoria A (Nobre -> Silo A)      | 10            | 18.70               | 12.47                      
Categoria B (Secundário -> Silo B) | 9             | 16.30               | 10.87                      
Categoria C (Rejeito -> Silo C)    | 10            | 17.00               | 11.33                      
